# ChatBot TODO z NLP (Slack)

Notatnik zawiera całe opracowanie chatbota. Wykonaną według załozeń 1. konserwatoriów

## 1. Biblioteki

Poza typowym importem bibliotek, przy pierwszym uruchomieniu NLTK pobiera dane potrzebne do tokenizacji.

In [1]:
import html
import os
import re
import shlex
import sqlite3
import unicodedata
import uuid
from datetime import datetime

import nltk
import numpy as np
import pandas as pd
import spacy
from dotenv import load_dotenv
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from pystempel import Stemmer
from sentence_transformers import SentenceTransformer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from slack_bolt import App
from slack_bolt.adapter.socket_mode import SocketModeHandler
from spacy.lang.pl.stop_words import STOP_WORDS

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab")

W1020 15:09:49.175000 17976 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 2. Konfiguracja

Wczytywanie kodów dostępów do Slacka zdefiniowanych w pliku środowiskowym `.env`.

In [2]:
load_dotenv()
ALBUM_NUMBER = os.getenv("ALBUM_NUMBER", "64823")

## 3. Baza danych

Zadania zapisujemy w pliku `tasks.db`. Każde zadanie ma id (UUID), treść, opcjonalną relację
do zadania nadrzędnego, status, daty utworzenia i modyfikacji oraz etykietę klasy (`label`) używaną w klasyfikacji. Kazda z operacji ma swoją osobną metodę z query do bazy danych.

In [3]:
DB_FILE = "tasks.db"


def connect():
    conn = sqlite3.connect(DB_FILE)
    conn.row_factory = sqlite3.Row 
    return conn


def init_db():
    with connect() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS tasks (
                id          TEXT PRIMARY KEY,
                description TEXT NOT NULL,
                relation    TEXT,
                status      TEXT NOT NULL,
                created_at  TEXT NOT NULL,
                updated_at  TEXT NOT NULL,
                label       TEXT
            )
        """)
        try:
            conn.execute("ALTER TABLE tasks ADD COLUMN label TEXT")
        except sqlite3.OperationalError:
            pass  # kolumna już istnieje


def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def db_add_task(description, relation=None):
    task_id = str(uuid.uuid4())
    with connect() as conn:
        conn.execute(
            "INSERT INTO tasks (id, description, relation, status, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?)",
            (task_id, description, relation, "todo", now(), now()),
        )
    return task_id


def db_get_all_tasks():
    with connect() as conn:
        return conn.execute("SELECT * FROM tasks ORDER BY created_at").fetchall()


def db_get_task(task_id):
    with connect() as conn:
        return conn.execute("SELECT * FROM tasks WHERE id = ?", (task_id,)).fetchone()


def db_update_description(task_id, description):
    with connect() as conn:
        conn.execute(
            "UPDATE tasks SET description = ?, updated_at = ? WHERE id = ?",
            (description, now(), task_id),
        )


def db_update_status(task_id, status):
    with connect() as conn:
        conn.execute(
            "UPDATE tasks SET status = ?, updated_at = ? WHERE id = ?",
            (status, now(), task_id),
        )


def db_update_label(task_id, label):
    with connect() as conn:
        conn.execute("UPDATE tasks SET label = ? WHERE id = ?", (label, task_id))


def db_delete_task(task_id):
    with connect() as conn:
        conn.execute("DELETE FROM tasks WHERE id = ?", (task_id,))
        # podzadania usuniętego zadania tracą relację
        conn.execute("UPDATE tasks SET relation = NULL WHERE relation = ?", (task_id,))


init_db()

## 4. Operacje NLP

Każdą operację najpierw definiujemy, a potem sprawdzamy na przykładowym zdaniu.
Na początku ładujemy model spaCy i stemmer Stempel.

In [4]:
nlp_pl = spacy.load("pl_core_news_sm")
stemmer = Stemmer.default()

example = "Przygotować prezentację z przetwarzania języka naturalnego"

Loading: 100%|██████████| 2225192/2225192 [00:00<00:00, 4515167.76bytes/s]


### Tokenizacja

Tokenizacja dzieli tekst na pojedyncze elementy (słowa, liczby, znaki). Używamy dwóch metod, NLTK i spaCy,
które różnią się np. obsługą skrótów i dat.

In [5]:
def tokenize_nltk(text):
    return word_tokenize(text, language="polish")


def tokenize_spacy(text):
    return [token.text for token in nlp_pl(text)]


print("NLTK: ", tokenize_nltk(example))
print("spaCy:", tokenize_spacy(example))

NLTK:  ['Przygotować', 'prezentację', 'z', 'przetwarzania', 'języka', 'naturalnego']
spaCy: ['Przygotować', 'prezentację', 'z', 'przetwarzania', 'języka', 'naturalnego']


### Czyszczenie tekstu

Funkcja `clean_text` zostawia w tekście same słowa. Usuwa liczby, interpunkcję, znaki specjalne,
pojedyncze litery i zbędne spacje.

In [6]:
def clean_text(text):
    text = text.lower()                       # małe litery
    text = re.sub(r"\d+", " ", text)          # usunięcie liczb
    text = re.sub(r"[^\w\s]|_", " ", text)    # usunięcie interpunkcji i znaków specjalnych
    text = re.sub(r"\b\w\b", " ", text)       # usunięcie pojedynczych znaków
    text = re.sub(r"\s+", " ", text)          # usunięcie nadmiarowych białych znaków
    return text.strip()


clean_text(example)

'przygotować prezentację przetwarzania języka naturalnego'

### Normalizacja tekstu

Normalizacja ujednolica zapis tekstu, ale nie usuwa słów.

In [7]:
def normalize_text(text, remove_diacritics=False):
    text = unicodedata.normalize("NFKC", text)  # ujednolicenie znaków Unicode
    text = text.replace("„", '"').replace("”", '"').replace("–", "-").replace("—", "-")
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()    # normalizacja białych znaków
    if remove_diacritics:
        text = text.replace("ł", "l")  # "ł" nie rozkłada się na literę + znak diakrytyczny
        text = unicodedata.normalize("NFKD", text)
        text = "".join(c for c in text if not unicodedata.combining(c))
    return text


print(normalize_text(example))
print(normalize_text(example, remove_diacritics=True))

przygotować prezentację z przetwarzania języka naturalnego
przygotowac prezentacje z przetwarzania jezyka naturalnego


### N-gramy

Rozmiar x-gramu zależy od ostatniej cyfry numeru albumu: cyfry 4–9 to od razu x, a do cyfr 0–3 dodajemy 4.

In [8]:
def get_ngrams(text, n):
    tokens = [t for t in tokenize_nltk(normalize_text(text)) if t.isalnum()]
    return list(ngrams(tokens, n))


def get_x(album_number):
    last_digit = int(album_number[-1])
    if last_digit >= 4:
        return last_digit
    return last_digit + 4


x = get_x(ALBUM_NUMBER)
print("Unigramy:", get_ngrams(example, 1))
print("Bigramy: ", get_ngrams(example, 2))
print("Trigramy:", get_ngrams(example, 3))
print(f"{x}-gramy: ", get_ngrams(example, x))

Unigramy: [('przygotować',), ('prezentację',), ('z',), ('przetwarzania',), ('języka',), ('naturalnego',)]
Bigramy:  [('przygotować', 'prezentację'), ('prezentację', 'z'), ('z', 'przetwarzania'), ('przetwarzania', 'języka'), ('języka', 'naturalnego')]
Trigramy: [('przygotować', 'prezentację', 'z'), ('prezentację', 'z', 'przetwarzania'), ('z', 'przetwarzania', 'języka'), ('przetwarzania', 'języka', 'naturalnego')]
7-gramy:  []


### Bag of Words

Bag of Words zlicza, ile razy każde słowo występuje w tekście. Słownik budujemy ze wszystkich zadań,
a wynik pokazujemy dla jednego z nich.

In [9]:
BOW_PARAMS = {
    "lowercase": True,
    "ngram_range": (1, 2),
    "max_features": 50,
    "min_df": 1,
    "max_df": 1.0,
    "binary": False,
}


def bag_of_words(texts, index):
    vectorizer = CountVectorizer(**BOW_PARAMS)
    matrix = vectorizer.fit_transform(texts)
    words = vectorizer.get_feature_names_out()
    row = matrix.toarray()[index]
    return {word: int(value) for word, value in zip(words, row) if value > 0}


example_tasks = [
    example,
    "Przygotować raport z laboratorium",
    "Kupić mleko i chleb",
]
bag_of_words(example_tasks, 1)

{'laboratorium': 1,
 'przygotować': 1,
 'przygotować raport': 1,
 'raport': 1,
 'raport laboratorium': 1}

### TF-IDF

TF-IDF też liczy słowa, ale obniża wagę tych, które pojawiają się w wielu zadaniach.
Dzięki temu najwyższe wagi mają słowa charakterystyczne dla danego zadania.

In [10]:
TFIDF_PARAMS = {
    "lowercase": True,
    "ngram_range": (1, 2),
    "max_features": 50,
    "min_df": 1,
    "max_df": 1.0,
    "use_idf": True,
    "smooth_idf": True,
    "sublinear_tf": True,
}


def tfidf(texts, index):
    vectorizer = TfidfVectorizer(**TFIDF_PARAMS)
    matrix = vectorizer.fit_transform(texts)
    words = vectorizer.get_feature_names_out()
    row = matrix.toarray()[index]
    return {word: round(float(value), 3) for word, value in zip(words, row) if value > 0}


tfidf(example_tasks, 1)

{'laboratorium': 0.467,
 'przygotować': 0.355,
 'przygotować raport': 0.467,
 'raport': 0.467,
 'raport laboratorium': 0.467}

### Stemming

Stemming obcina końcówki słów i zostawia ich rdzeń. NLTK nie ma stemmera dla polskiego, więc używamy algorytmu Stempel.

In [11]:
def stem(text):
    words = [t.lower() for t in tokenize_nltk(text) if t.isalpha()]
    result = []
    for word in words:
        # dla bardzo krótkich słów (np. "z", "do") Stempel zwraca bzdury, więc je pomijamy
        stemmed = stemmer(word) if len(word) > 2 else word
        result.append(stemmed or word)
    return result


stem(example)

['przygotować', 'prezentacja', 'z', 'przetwarzać', 'języka', 'naturalny']

### Lematyzacja

Lematyzacja zamienia słowo na jego formę słownikową, np. „języka” → „język”. Robi to polski model spaCy.

In [12]:
def lemmatize(text):
    return [token.lemma_ for token in nlp_pl(text) if not token.is_punct]


lemmatize(example)

['przygotować', 'prezentacja', 'z', 'przetwarzać', 'język', 'naturalny']

### Usuwanie stopwords

Stopwords to częste słowa, które niewiele wnoszą, np. „z”, „i”, „do”. Polską listę bierzemy ze spaCy, bo NLTK jej nie ma.

In [13]:
def remove_stopwords(text):
    words = [t for t in tokenize_nltk(text) if t.isalnum()]
    return " ".join(w for w in words if w.lower() not in STOP_WORDS)


print("Oryginał:     ", example)
print("Bez stopwords:", remove_stopwords(example))

Oryginał:      Przygotować prezentację z przetwarzania języka naturalnego
Bez stopwords: Przygotować prezentację przetwarzania języka naturalnego


## 5. Komendy bota

Każda komenda to zwykła funkcja, która dostaje tekst wpisany po komendzie i zwraca odpowiedź.

In [14]:
STATUSES = ["todo", "in_progress", "done"]
OPERATIONS = [
    "tokens", "clean", "normalize", "unigram", "bigram", "trigram", "xgram",
    "bow", "tfidf", "stem", "lemma", "stopwords",
]


def tasks_command():
    tasks = db_get_all_tasks()
    if not tasks:
        return "Brak zadań."
    lines = []
    for task in tasks:
        line = f"• {task['description']}\n    id: `{task['id']}` | status: {task['status']}"
        if task["relation"]:
            line += f" | zadanie nadrzędne: `{task['relation']}`"
        if task["label"]:
            line += f" | etykieta: {task['label']}"
        lines.append(line)
    return "\n".join(lines)


def add_command(text):
    description, _, relation = text.partition("|")
    description = description.strip()
    relation = relation.strip() or None
    if not description:
        return "Użycie: /add [treść zadania] | [id zadania nadrzędnego - opcjonalnie]"
    if relation and db_get_task(relation) is None:
        return "Nie ma zadania nadrzędnego o takim id."
    task_id = db_add_task(description, relation)
    return f"Utworzono zadanie: `{task_id}`"


def update_command(text):
    parts = text.split(maxsplit=1)
    if len(parts) < 2:
        return "Użycie: /update [id] [nowa treść] albo /update [id] status=done"
    task_id, new_value = parts
    if db_get_task(task_id) is None:
        return "Nie znaleziono zadania o takim id."

    if new_value.startswith("status="):
        status = new_value.removeprefix("status=").strip()
        if status not in STATUSES:
            return "Dozwolone statusy: " + ", ".join(STATUSES)
        db_update_status(task_id, status)
    else:
        db_update_description(task_id, new_value)
    return f"Zaktualizowano zadanie `{task_id}`"


def delete_command(text):
    task_id = text.strip()
    if db_get_task(task_id) is None:
        return "Nie znaleziono zadania o takim id."
    db_delete_task(task_id)
    return f"Usunięto zadanie `{task_id}`"


def task_command(text):
    parts = text.split()
    if len(parts) != 2:
        return "Użycie: /task [id] [operacja]\nOperacje: " + ", ".join(OPERATIONS)
    task_id, operation = parts
    task = db_get_task(task_id)
    if task is None:
        return "Nie znaleziono zadania o takim id."

    description = task["description"]
    all_tasks = db_get_all_tasks()
    texts = [t["description"] for t in all_tasks]
    index = [t["id"] for t in all_tasks].index(task_id)

    if operation == "tokens":
        nltk_tokens = tokenize_nltk(description)
        spacy_tokens = tokenize_spacy(description)
        same = "tak" if nltk_tokens == spacy_tokens else "nie"
        result = f"NLTK:  {nltk_tokens}\nspaCy: {spacy_tokens}\nWyniki identyczne: {same}"
    elif operation == "clean":
        result = clean_text(description)
    elif operation == "normalize":
        result = (
            f"{normalize_text(description)}\n"
            f"Bez znaków diakrytycznych: {normalize_text(description, remove_diacritics=True)}"
        )
    elif operation == "unigram":
        result = get_ngrams(description, 1)
    elif operation == "bigram":
        result = get_ngrams(description, 2)
    elif operation == "trigram":
        result = get_ngrams(description, 3)
    elif operation == "xgram":
        x = get_x(ALBUM_NUMBER)
        result = f"x = {x}\n{get_ngrams(description, x)}"
    elif operation == "bow":
        result = f"Parametry: {BOW_PARAMS}\n{bag_of_words(texts, index)}"
    elif operation == "tfidf":
        result = f"Parametry: {TFIDF_PARAMS}\n{tfidf(texts, index)}"
    elif operation == "stem":
        result = stem(description)
    elif operation == "lemma":
        result = lemmatize(description)
    elif operation == "stopwords":
        result = f"Tekst oryginalny: {description}\nBez stopwords:    {remove_stopwords(description)}"
    else:
        return "Nieznana operacja. Dostępne: " + ", ".join(OPERATIONS)

    return f"```{result}```"

### Test komend

Dodajemy przykładowe zadanie, sprawdzamy na nim kilka komend, a na końcu je usuwamy.

In [15]:
answer = add_command("Przygotować prezentację z przetwarzania języka naturalnego")
print(answer)
test_id = answer.split("`")[1]

print(tasks_command())
print(task_command(f"{test_id} tokens"))
print(task_command(f"{test_id} bigram"))
print(update_command(f"{test_id} status=done"))
print(delete_command(test_id))

Utworzono zadanie: `361fc6c4-1df8-40d3-8098-6b8e978bfbef`
• Przygotować prezentację z przetwarzania języka naturalnego
    id: `ed665d65-d36a-44ed-9870-fa3636adeb57` | status: todo | etykieta: nauka
• in_progress
    id: `79a031fc-0851-4358-b363-4abee4eba9c0` | status: todo
• Wysłać raport sprzedaży do szefa
    id: `68fd1683-37f9-48ef-9aee-c3be79309b7a` | status: todo | etykieta: praca
• Przygotować ofertę dla nowego klienta
    id: `67427194-6d33-4d1d-bcf1-bede67dc03d6` | status: todo | etykieta: praca
• Spotkanie z zespołem o godzinie 10
    id: `01086cc6-d006-4ab9-ad26-ac0f3c4d6134` | status: todo | etykieta: praca
• Odpisać na maile od klientów
    id: `38bd43eb-f2ef-4d8e-907e-d57b5192ebe9` | status: todo | etykieta: praca
• Wystawić fakturę za październik
    id: `27844e15-ad65-40c5-8991-d5149c8909b5` | status: todo | etykieta: praca
• Podpisać umowę z dostawcą
    id: `53e10297-cd90-4113-bac4-f4be4085d408` | status: todo | etykieta: praca
• Zaplanować sprint dla zespołu
    id: 

## 6. Klasyfikacja tekstu

Bot uczy klasyfikatory na dwóch zbiorach: zadaniach TODO z własnymi etykietami (`praca`, `dom`, `nauka`) oraz 4 tematach z 20 Newsgroups.
Nazwisko zaczyna się na K, więc metody to dwie alfabetycznie najbliższe: **Linear SVM** i **Logistic Regression**.

In [16]:
SAMPLE_TASKS = [
    "Wysłać raport sprzedaży do szefa",
    "Przygotować ofertę dla nowego klienta",
    "Spotkanie z zespołem o godzinie 10",
    "Odpisać na maile od klientów",
    "Wystawić fakturę za październik",
    "Podpisać umowę z dostawcą",
    "Zaplanować sprint dla zespołu",
    "Poprawić błąd w kodzie aplikacji przed wdrożeniem",
    "Przygotować budżet działu na przyszły rok",
    "Zadzwonić do klienta w sprawie reklamacji",
    "Zarezerwować salę na spotkanie z zarządem",
    "Uzupełnić raport godzin w systemie firmy",
    "Przygotować prezentację dla klienta",
    "Zrobić przegląd kodu kolegi z zespołu",
    "Rozliczyć delegację służbową",
    "Omówić z szefem podwyżkę",
    "Zaktualizować dokumentację projektu w firmie",
    "Wysłać umowę do działu prawnego",
    "Przeprowadzić rozmowę rekrutacyjną z kandydatem",
    "Przygotować agendę spotkania z klientem",
    "Sprawdzić wyniki kampanii marketingowej firmy",
    "Wdrożyć nową wersję aplikacji na serwer",
    "Napisać podsumowanie kwartału dla zarządu",
    "Zamówić nowe laptopy do biura",
    "Odpowiedzieć na zapytanie ofertowe",

    "Kupić mleko i chleb",
    "Posprzątać kuchnię",
    "Zrobić pranie",
    "Zapłacić rachunek za prąd",
    "Ugotować obiad dla rodziny",
    "Wynieść śmieci",
    "Odkurzyć mieszkanie",
    "Wyprowadzić psa na spacer",
    "Kupić karmę dla kota",
    "Naprawić kran w łazience",
    "Zapłacić czynsz za mieszkanie",
    "Umyć okna w salonie",
    "Zrobić zakupy na weekend",
    "Podlać kwiaty w ogrodzie",
    "Rozmrozić lodówkę",
    "Wyprasować koszule",
    "Wymienić żarówkę w przedpokoju",
    "Zrobić listę zakupów",
    "Posprzątać łazienkę",
    "Skosić trawę w ogrodzie",
    "Wezwać hydraulika do mieszkania",
    "Wypakować zmywarkę",
    "Kupić prezent dla mamy",
    "Upiec ciasto na niedzielny obiad",
    "Zmienić pościel w sypialni",

    "Przygotować się do kolokwium z matematyki",
    "Napisać sprawozdanie z laboratorium",
    "Powtórzyć materiał do egzaminu ze statystyki",
    "Przeczytać rozdział podręcznika do NLP",
    "Oddać projekt zaliczeniowy prowadzącemu",
    "Zrobić notatki z wykładu",
    "Rozwiązać zadania z algebry",
    "Nauczyć się słówek z angielskiego",
    "Przygotować referat na seminarium",
    "Zapisać się na egzamin poprawkowy",
    "Wypożyczyć książkę z biblioteki uczelni",
    "Obejrzeć nagranie wykładu z programowania",
    "Napisać esej na zajęcia z filozofii",
    "Przygotować prezentację z przetwarzania języka naturalnego",
    "Zrobić ćwiczenia z analizy matematycznej",
    "Wysłać pracę domową prowadzącemu",
    "Powtórzyć wzory do kolokwium z fizyki",
    "Przeczytać artykuł naukowy o sieciach neuronowych",
    "Uzupełnić notatki z ćwiczeń",
    "Zapisać się na konsultacje do wykładowcy",
    "Napisać rozdział pracy inżynierskiej",
    "Rozwiązać test próbny przed egzaminem",
    "Przygotować się do zajęć z baz danych",
    "Odrobić zadanie z programowania w Pythonie",
    "Zrobić fiszki do nauki historii",
]

existing = {task["description"] for task in db_get_all_tasks()}
added = 0
for description in SAMPLE_TASKS:
    if description not in existing:
        db_add_task(description)
        added += 1
print("Dodano zadań:", added)

Dodano zadań: 0


### Etykietowanie zadań TODO

Klasę zadania wybieramy po słowach kluczowych: liczymy, ile lematów zadania pasuje do każdej klasy, i wybieramy tę z największą liczbą trafień.
Etykieta trafia do kolumny `label`, a ponowne etykietowanie nadpisuje stare wartości.

In [17]:
LABEL_KEYWORDS = {
    "praca": [
        "raport", "szef", "klient", "oferta", "ofertowy", "spotkanie", "zespół", "maile", "faktura",
        "umowa", "sprint", "kod", "aplikacja", "wdrożyć", "budżet", "dział", "reklamacja", "zarząd",
        "firma", "delegacja", "służbowy", "podwyżka", "dokumentacja", "rekrutacyjny", "kandydat",
        "agenda", "kampania", "serwer", "kwartał", "biuro",
    ],
    "dom": [
        "mleko", "chleb", "kupić", "posprzątać", "kuchnia", "prać", "rachunek", "obiad", "rodzina",
        "śmieć", "odkurzyć", "mieszkanie", "mieszkać", "pies", "kot", "karma", "kran", "łazienka",
        "czynsz", "okno", "salon", "zakupy", "zakup", "kwiat", "ogród", "lodówka", "wyprasować",
        "koszula", "żarówka", "przedpokój", "trawa", "hydraulik", "zmywarka", "prezent", "ciasto",
        "pościel", "sypialnia",
    ],
    "nauka": [
        "kolokwium", "matematyka", "matematyczny", "sprawozdanie", "laboratorium", "egzamin",
        "statystyka", "rozdział", "podręcznik", "nlp", "projekt", "zaliczeniowy", "prowadzący",
        "prowadzącemu", "notatka", "wykład", "zadanie", "algebra", "nauczyć", "angielski", "referat",
        "seminarium", "książka", "biblioteka", "uczelnia", "esej", "zajęcia", "filozofia", "język",
        "ćwiczenie", "domowy", "fizyka", "wzór", "artykuł", "naukowy", "konsultacja", "wykładowca",
        "inżynierski", "test", "programowania", "nauka", "fiszka", "historia",
    ],
}


def assign_label(text):
    lemmas = [lemma.lower() for lemma in lemmatize(text)]
    scores = {label: sum(lemma in words for lemma in lemmas) for label, words in LABEL_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else None


def prepare_todo():
    texts, labels = [], []
    for task in db_get_all_tasks():
        label = assign_label(task["description"])
        db_update_label(task["id"], label)
        if label:
            texts.append(clean_text(task["description"]))
            labels.append(label)
    return texts, labels


assign_label("Napisać raport z laboratorium")

'praca'

### Zbiór 20 Newsgroups

Z 20 Newsgroups bierzemy 4 tematy: grafikę komputerową, hokej, medycynę i kosmos.
Usuwamy nagłówki, podpisy i cytaty, żeby model uczył się z treści wiadomości, a nie np. z adresu nadawcy.

In [18]:
NEWSGROUPS = {
    "comp.graphics": "grafika",
    "rec.sport.hockey": "hokej",
    "sci.med": "medycyna",
    "sci.space": "kosmos",
}


def prepare_newsgroups():
    news = fetch_20newsgroups(subset="all", categories=list(NEWSGROUPS), remove=("headers", "footers", "quotes"))
    texts, labels = [], []
    for document, target in zip(news.data, news.target):
        cleaned = clean_text(document)
        if len(cleaned.split()) >= 5:  # pomijamy puste i bardzo krótkie wiadomości
            texts.append(cleaned)
            labels.append(NEWSGROUPS[news.target_names[target]])
    return texts, labels


news_texts, news_labels = prepare_newsgroups()
print("Liczba dokumentów:", len(news_texts))
pd.Series(news_labels).value_counts()

Liczba dokumentów: 3807


hokej       962
medycyna    953
kosmos      948
grafika     944
Name: count, dtype: int64

In [19]:
CLASS_BOW_PARAMS = {"ngram_range": (1, 2), "max_features": 5000, "min_df": 1, "max_df": 0.9}
CLASS_TFIDF_PARAMS = {**CLASS_BOW_PARAMS, "sublinear_tf": True}

encoder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embedding_cache = {}


def embed(texts):
    # liczenie embeddingów jest wolne, więc zapamiętujemy już policzone
    missing = [text for text in texts if text not in embedding_cache]
    if missing:
        for text, vector in zip(missing, encoder.encode(missing, batch_size=64)):
            embedding_cache[text] = vector
    return np.array([embedding_cache[text] for text in texts])


embed(["przygotować prezentację"]).shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(1, 384)

In [20]:
REPRESENTATIONS = ["bow", "tfidf", "embedding"]
METHODS = ["svm", "logreg"]


def make_classifier(method_class):
    if method_class == "svm":
        return CalibratedClassifierCV(LinearSVC(C=1.0))
    return LogisticRegression(max_iter=1000)

In [21]:
datasets = {}
models = {}
active_dataset = None


def train_model(method_repr, method_class):
    texts, labels = datasets[active_dataset]
    X_train, X_test, y_train, y_test = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=42
    )
    vectorizer = None
    if method_repr == "embedding":
        X_train_vec, X_test_vec = embed(X_train), embed(X_test)
    else:
        if method_repr == "bow":
            vectorizer = CountVectorizer(**CLASS_BOW_PARAMS)
        else:
            vectorizer = TfidfVectorizer(**CLASS_TFIDF_PARAMS)
        X_train_vec = vectorizer.fit_transform(X_train)  # słownik tylko ze zbioru treningowego
        X_test_vec = vectorizer.transform(X_test)
    model = make_classifier(method_class).fit(X_train_vec, y_train)
    return {"model": model, "vectorizer": vectorizer, "X_test": X_test_vec, "y_test": y_test}


def get_model(method_repr, method_class):
    key = (active_dataset, method_repr, method_class)
    if key not in models:
        models[key] = train_model(method_repr, method_class)
    return models[key]


def vectorize(texts, trained):
    if trained["vectorizer"] is None:
        return embed(texts)
    return trained["vectorizer"].transform(texts)


def compute_metrics(cm, classes):
    total = cm.sum()
    rows = []
    for i, label in enumerate(classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        precision = tp / (tp + fp) if tp + fp else 0
        recall = tp / (tp + fn) if tp + fn else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
        rows.append({
            "klasa": label, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "ACC": (tp + tn) / total, "ERR": (fp + fn) / total,
            "Precision": precision, "Recall": recall, "F1": f1, "P": tp + fn,
        })
    return pd.DataFrame(rows).set_index("klasa")


def evaluate(method_repr, method_class):
    trained = get_model(method_repr, method_class)
    predictions = trained["model"].predict(trained["X_test"])
    classes = sorted(set(trained["y_test"]))
    cm = confusion_matrix(trained["y_test"], predictions, labels=classes)
    per_class = compute_metrics(cm, classes)
    weights = per_class["P"] / per_class["P"].sum()
    acc = np.trace(cm) / cm.sum()  # w wieloklasowej ACC to przekątna macierzy / wszystkie przykłady
    summary = {
        "ACC": acc,
        "ERR": 1 - acc,
        "Precision (macro)": per_class["Precision"].mean(),
        "Recall (macro)": per_class["Recall"].mean(),
        "F1 (macro)": per_class["F1"].mean(),
        "F1 (weighted)": (per_class["F1"] * weights).sum(),
    }
    matrix = pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes])
    return matrix, per_class, summary

### Komendy klasyfikacji

In [22]:
def prepare_command(text):
    global active_dataset
    name = text.strip().lower() or "todo"
    if name == "todo":
        datasets[name] = prepare_todo()
    elif name in ("20newsgroups", "20newsgroup"):
        name = "20newsgroups"
        datasets[name] = prepare_newsgroups()
    else:
        return "Dostępne zbiory: todo, 20newsgroups"
    active_dataset = name
    for key in [key for key in models if key[0] == name]:
        del models[key]  # nowe etykiety = modele trzeba nauczyć od nowa
    texts, labels = datasets[name]
    return f"Przypisano etykiety do {len(labels)} wpisów. Klasy: {', '.join(sorted(set(labels)))}"


def classifier_command(text):
    if active_dataset is None:
        return "Najpierw użyj /prepare_data_for_classification [todo|20newsgroups]"
    usage = 'Użycie: /classifier cleaning "tekst" [bow|tfidf|embedding] [svm|logreg]'
    text = text.replace("„", '"').replace("“", '"').replace("”", '"')  # Slack/macOS zamienia cudzysłowy
    try:
        parts = shlex.split(text)
    except ValueError:
        return usage
    if len(parts) != 4 or parts[0] != "cleaning" or parts[2] not in REPRESENTATIONS or parts[3] not in METHODS:
        return usage

    _, sentence, method_repr, method_class = parts
    trained = get_model(method_repr, method_class)
    probabilities = trained["model"].predict_proba(vectorize([clean_text(sentence)], trained))[0]
    best = probabilities.argmax()
    label = trained["model"].classes_[best]
    return (
        f"Przewidziana klasa: *{label}* (p={probabilities[best]:.2f}) | "
        f"zbiór: {active_dataset} | repr: {method_repr} | metoda: {method_class}"
    )


def evaluate_command(text):
    if active_dataset is None:
        return "Najpierw użyj /prepare_data_for_classification [todo|20newsgroups]"
    parts = text.split()
    if len(parts) != 2 or parts[0] not in REPRESENTATIONS or parts[1] not in METHODS:
        return "Użycie: /evaluate [bow|tfidf|embedding] [svm|logreg]"

    matrix, per_class, summary = evaluate(parts[0], parts[1])
    return (
        f"Zbiór: {active_dataset} | repr: {parts[0]} | metoda: {parts[1]}\n"
        f"```{matrix.to_string()}\n\n{per_class.round(2).to_string()}\n\n"
        f"ACC = {summary['ACC']:.2f} | ERR = {summary['ERR']:.2f} | "
        f"F1 (macro) = {summary['F1 (macro)']:.2f} | F1 (weighted) = {summary['F1 (weighted)']:.2f}```"
    )

### Test komend klasyfikacji

In [23]:
print(prepare_command("todo"))
print(classifier_command('cleaning "Napisać raport z laboratorium" tfidf logreg'))
print(evaluate_command("tfidf logreg"))

Przypisano etykiety do 75 wpisów. Klasy: dom, nauka, praca
Przewidziana klasa: *nauka* (p=0.41) | zbiór: todo | repr: tfidf | metoda: logreg
Zbiór: todo | repr: tfidf | metoda: logreg
```            pred_dom  pred_nauka  pred_praca
true_dom           4           0           1
true_nauka         1           4           0
true_praca         1           1           3

       TP  FP  FN  TN   ACC   ERR  Precision  Recall    F1  P
klasa                                                        
dom     4   2   1   8  0.80  0.20       0.67     0.8  0.73  5
nauka   4   1   1   9  0.87  0.13       0.80     0.8  0.80  5
praca   3   1   2   9  0.80  0.20       0.75     0.6  0.67  5

ACC = 0.73 | ERR = 0.27 | F1 (macro) = 0.73 | F1 (weighted) = 0.73```


### Porównanie 6 konfiguracji

Dla każdego zbioru uczymy wszystkie konfiguracje (3 reprezentacje × 2 metody) i wypisujemy macierze pomyłek.
Na końcu zbieramy ACC, ERR i F1 w jednej tabeli.

In [24]:
results = []
for name in ["todo", "20newsgroups"]:
    prepare_command(name)
    for method_repr in REPRESENTATIONS:
        for method_class in METHODS:
            matrix, per_class, summary = evaluate(method_repr, method_class)
            print(f"=== {name} | {method_repr} | {method_class}")
            print(matrix, "\n")
            results.append({"zbiór": name, "repr": method_repr, "metoda": method_class, **summary})

pd.DataFrame(results).round(3)

=== todo | bow | svm
            pred_dom  pred_nauka  pred_praca
true_dom           4           0           1
true_nauka         2           2           1
true_praca         1           1           3 

=== todo | bow | logreg
            pred_dom  pred_nauka  pred_praca
true_dom           5           0           0
true_nauka         3           1           1
true_praca         2           1           2 

=== todo | tfidf | svm
            pred_dom  pred_nauka  pred_praca
true_dom           4           0           1
true_nauka         2           3           0
true_praca         1           1           3 

=== todo | tfidf | logreg
            pred_dom  pred_nauka  pred_praca
true_dom           4           0           1
true_nauka         1           4           0
true_praca         1           1           3 

=== todo | embedding | svm
            pred_dom  pred_nauka  pred_praca
true_dom           5           0           0
true_nauka         0           5           0
true_praca      

/Users/adam.kostuch/Desktop/private/university/chatbot/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/adam.kostuch/Desktop/private/university/chatbot/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/adam.kostuch/Desktop/private/university/chatbot/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


=== 20newsgroups | bow | svm
               pred_grafika  pred_hokej  pred_kosmos  pred_medycyna
true_grafika            154           6           17             12
true_hokej                7         172            5              8
true_kosmos              12           8          153             17
true_medycyna            12           6           12            161 

=== 20newsgroups | bow | logreg
               pred_grafika  pred_hokej  pred_kosmos  pred_medycyna
true_grafika            156           5           19              9
true_hokej                5         172            7              8
true_kosmos              11           7          159             13
true_medycyna            10           3           14            164 

=== 20newsgroups | tfidf | svm
               pred_grafika  pred_hokej  pred_kosmos  pred_medycyna
true_grafika            169           4           11              5
true_hokej                4         180            1              7
true_kosmos         

,zbiór,repr,metoda,ACC,ERR,Precision (macro),Recall (macro),F1 (macro),F1 (weighted)
0,todo,bow,svm,0.600,0.400,0.613,0.600,0.589,0.589
1,todo,bow,logreg,0.533,0.467,0.556,0.533,0.484,0.484
2,todo,tfidf,svm,0.667,0.333,0.690,0.667,0.667,0.667
3,todo,tfidf,logreg,0.733,0.267,0.739,0.733,0.731,0.731
4,todo,embedding,svm,1.000,0.000,1.000,1.000,1.000,1.000
5,todo,embedding,logreg,1.000,0.000,1.000,1.000,1.000,1.000
6,20newsgroups,bow,svm,0.840,0.160,0.840,0.840,0.840,0.840
7,20newsgroups,bow,logreg,0.854,0.146,0.855,0.854,0.855,0.855
8,20newsgroups,tfidf,svm,0.916,0.084,0.916,0.916,0.916,0.916
9,20newsgroups,tfidf,logreg,0.899,0.101,0.901,0.899,0.899,0.899


### Walidacja na własnych zdaniach

Sprawdzamy klasyfikatory na zdaniach, których nie ma w żadnym zbiorze, i porównujemy przewidywania z klasą, której się spodziewamy.

In [25]:
OWN_SENTENCES = {
    "todo": [
        ("Zadzwonić do kontrahenta w sprawie zamówienia", "praca"),
        ("Spotkać się z przełożonym w sprawie awansu", "praca"),
        ("Przygotować prezentację wyników dla klienta", "praca"),
        ("Posprzątać garaż i piwnicę", "dom"),
        ("Kupić nowe ręczniki do łazienki", "dom"),
        ("Zarezerwować stolik w restauracji na urodziny żony", "dom"),
        ("Przeczytać notatki przed sesją", "nauka"),
        ("Zapisać się na kurs uczenia maszynowego", "nauka"),
        ("Przygotować slajdy na obronę pracy dyplomowej", "nauka"),
    ],
    "20newsgroups": [
        ("I need a fast algorithm to render 3D images with shading", "grafika"),
        ("Which file format is best for storing JPEG and GIF pictures", "grafika"),
        ("My new graphics card shows strange colors on the monitor", "grafika"),
        ("The goalie stopped every shot in the final period", "hokej"),
        ("Who will win the Stanley Cup playoffs this season", "hokej"),
        ("The team traded their best defenseman before the game", "hokej"),
        ("My doctor prescribed antibiotics for the infection", "medycyna"),
        ("Does this diet help patients with high blood pressure", "medycyna"),
        ("The vaccine caused side effects in some children", "medycyna"),
        ("NASA plans to launch a new satellite into orbit", "kosmos"),
        ("The astronauts repaired the telescope during the mission", "kosmos"),
        ("How much fuel does a rocket need to reach the Moon", "kosmos"),
    ],
}

for name, sentences in OWN_SENTENCES.items():
    prepare_command(name)
    table = pd.DataFrame({"oczekiwana": [label for _, label in sentences]}, index=[s for s, _ in sentences])
    for method_repr in REPRESENTATIONS:
        for method_class in METHODS:
            trained = get_model(method_repr, method_class)
            vectors = vectorize([clean_text(s) for s, _ in sentences], trained)
            table[f"{method_repr}/{method_class}"] = trained["model"].predict(vectors)
    correct = table.drop(columns="oczekiwana").eq(table["oczekiwana"], axis=0).sum()
    print(f"=== {name}: poprawne odpowiedzi na {len(sentences)} zdań")
    print(correct.to_string(), "\n")
    display(table)

=== todo: poprawne odpowiedzi na 9 zdań
bow/svm             7
bow/logreg          7
tfidf/svm           7
tfidf/logreg        7
embedding/svm       7
embedding/logreg    8 



,oczekiwana,bow/svm,bow/logreg,tfidf/svm,tfidf/logreg,embedding/svm,embedding/logreg
Zadzwonić do kontrahenta w sprawie zamówienia,praca,praca,praca,praca,praca,praca,praca
Spotkać się z przełożonym w sprawie awansu,praca,nauka,nauka,nauka,nauka,nauka,nauka
Przygotować prezentację wyników dla klienta,praca,praca,praca,praca,praca,praca,praca
Posprzątać garaż i piwnicę,dom,dom,dom,dom,dom,dom,dom
Kupić nowe ręczniki do łazienki,dom,dom,dom,dom,dom,dom,dom
Zarezerwować stolik w restauracji na urodziny żony,dom,praca,praca,praca,praca,praca,dom
Przeczytać notatki przed sesją,nauka,nauka,nauka,nauka,nauka,nauka,nauka
Zapisać się na kurs uczenia maszynowego,nauka,nauka,nauka,nauka,nauka,nauka,nauka
Przygotować slajdy na obronę pracy dyplomowej,nauka,nauka,nauka,nauka,nauka,nauka,nauka


/Users/adam.kostuch/Desktop/private/university/chatbot/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/adam.kostuch/Desktop/private/university/chatbot/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/adam.kostuch/Desktop/private/university/chatbot/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


=== 20newsgroups: poprawne odpowiedzi na 12 zdań
bow/svm             11
bow/logreg          12
tfidf/svm           12
tfidf/logreg        12
embedding/svm       12
embedding/logreg    12 



,oczekiwana,bow/svm,bow/logreg,tfidf/svm,tfidf/logreg,embedding/svm,embedding/logreg
I need a fast algorithm to render 3D images with shading,grafika,grafika,grafika,grafika,grafika,grafika,grafika
Which file format is best for storing JPEG and GIF pictures,grafika,grafika,grafika,grafika,grafika,grafika,grafika
My new graphics card shows strange colors on the monitor,grafika,grafika,grafika,grafika,grafika,grafika,grafika
The goalie stopped every shot in the final period,hokej,hokej,hokej,hokej,hokej,hokej,hokej
Who will win the Stanley Cup playoffs this season,hokej,hokej,hokej,hokej,hokej,hokej,hokej
The team traded their best defenseman before the game,hokej,hokej,hokej,hokej,hokej,hokej,hokej
My doctor prescribed antibiotics for the infection,medycyna,medycyna,medycyna,medycyna,medycyna,medycyna,medycyna
Does this diet help patients with high blood pressure,medycyna,medycyna,medycyna,medycyna,medycyna,medycyna,medycyna
The vaccine caused side effects in some children,medycyna,hokej,medycyna,medycyna,medycyna,medycyna,medycyna
NASA plans to launch a new satellite into orbit,kosmos,kosmos,kosmos,kosmos,kosmos,kosmos,kosmos


## 7. Uruchomienie bota w Slacku

Łączymy komendy ze Slackiem i uruchamiamy bota. Bot działa w tle, dopóki notatnik jest otwarty.

In [26]:
app = App(token=os.environ["SLACK_BOT_TOKEN"])


def get_text(command):
    return html.unescape(command.get("text", ""))


def reply(respond, text):
    respond(text=text, response_type="in_channel") 


@app.command("/tasks")
def handle_tasks(ack, respond):
    ack() 
    reply(respond, tasks_command())


@app.command("/add")
def handle_add(ack, respond, command):
    ack()
    reply(respond, add_command(get_text(command)))


@app.command("/update")
def handle_update(ack, respond, command):
    ack()
    reply(respond, update_command(get_text(command)))


@app.command("/delete")
def handle_delete(ack, respond, command):
    ack()
    reply(respond, delete_command(get_text(command)))


@app.command("/task")
def handle_task(ack, respond, command):
    ack()
    reply(respond, task_command(get_text(command)))


@app.command("/prepare_data_for_classification")
def handle_prepare(ack, respond, command):
    ack()
    reply(respond, prepare_command(get_text(command)))


@app.command("/classifier")
def handle_classifier(ack, respond, command):
    ack()
    reply(respond, classifier_command(get_text(command)))


@app.command("/evaluate")
def handle_evaluate(ack, respond, command):
    ack()
    reply(respond, evaluate_command(get_text(command)))

COMMANDS = {
    "tasks": lambda text: tasks_command(),
    "add": add_command,
    "update": update_command,
    "delete": delete_command,
    "task": task_command,
    "prepare_data_for_classification": prepare_command,
    "classifier": classifier_command,
    "evaluate": evaluate_command,
}

@app.event("message")
def handle_message(event, say):
    if event.get("channel_type") != "im" or event.get("bot_id") or event.get("subtype"):
        return
    text = html.unescape(event.get("text", "")).strip().lstrip("/")
    name, _, rest = text.partition(" ")
    if name in COMMANDS:
        say(COMMANDS[name](rest))
    else:
        say("Nieznana komenda. Dostępne: " + ", ".join(COMMANDS))


handler = SocketModeHandler(app, os.environ["SLACK_APP_TOKEN"])
handler.connect()
print("Bot działa!")

Bot działa!


Uruchom tę komórkę, aby zatrzymać bota.

In [ ]:
# handler.close()
# print("Bot zatrzymany.")

/Users/adam.kostuch/Desktop/private/university/chatbot/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/adam.kostuch/Desktop/private/university/chatbot/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/adam.kostuch/Desktop/private/university/chatbot/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
